# Part 1 Pipeline CLI Demo

This notebook demonstrates the new **production-style pipeline** through the CLI:

`generate-data -> train-ddp -> train-nn -> compare`

It keeps notebooks lightweight while moving reproducible execution into a config-driven command workflow.


## Why this notebook exists

- Show how to run the pipeline without manually wiring all training code in notebooks.
- Demonstrate **shared dataset reuse** across all methods (DDP + NN).
- Show artifact outputs (`dataset_id`, run manifest, checkpoints, figures).

This notebook uses a **smoke/demo config** by default so it runs quickly.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import os
import subprocess
from typing import Any, Dict

REPO_ROOT = Path("..").resolve()
CLI_PATH = REPO_ROOT / "scripts" / "run_part1_pipeline.py"
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "pipelines" / "part1_baseline.json"
DEMO_ROOT = REPO_ROOT / "results" / "pipeline_demo"

if not CLI_PATH.exists():
    raise FileNotFoundError(f"CLI not found: {CLI_PATH}")
if not BASE_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config not found: {BASE_CONFIG_PATH}")

print(f"Repo root: {REPO_ROOT}")
print(f"CLI path: {CLI_PATH}")
print(f"Base config: {BASE_CONFIG_PATH}")


## CLI helpers

`run_cli(...)` executes the pipeline command and prints stdout.
`extract_last_json(...)` parses the final JSON payload returned by CLI commands.


In [ ]:
def run_cli(args: list[str], timeout_sec: int = 3600) -> str:
    cmd = ["python", str(CLI_PATH), *args]
    print("$", " ".join(cmd))

    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")

    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout_sec,
        check=False,
    )

    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print("--- STDERR ---")
        print(proc.stderr)

    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")

    return proc.stdout


def extract_last_json(raw_text: str) -> Dict[str, Any]:
    raw = raw_text.strip()
    for i in range(len(raw) - 1, -1, -1):
        if raw[i] != "{":
            continue
        candidate = raw[i:]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue
    raise ValueError("No JSON object found in CLI output")


## Build a smoke/demo config

By default we reduce iteration counts so the demo can run on laptop quickly.
Set `USE_SMOKE_CONFIG = False` to use the full baseline config.


In [ ]:
USE_SMOKE_CONFIG = True

cfg = json.loads(BASE_CONFIG_PATH.read_text())

# Runtime training seed should be explicit and derived from the data master seed.
# This keeps pipeline runs deterministic end-to-end for a fixed config.
TRAINING_SEED = cfg["data"]["master_seed"]
for opt_key in ("basic_optimization", "risky_optimization"):
    cfg["nn"][opt_key]["training_seed"] = TRAINING_SEED
    cfg["nn"][opt_key]["deterministic_ops"] = True

if USE_SMOKE_CONFIG:
    cfg["output_root"] = str((DEMO_ROOT / "smoke").as_posix())

    # NN: fast smoke training
    cfg["nn"]["basic_optimization"]["n_iter"] = 20
    cfg["nn"]["basic_optimization"]["batch_size"] = 32
    cfg["nn"]["basic_optimization"]["log_every"] = 5
    cfg["nn"]["basic_optimization"]["early_stopping"]["enabled"] = False

    cfg["nn"]["risky_optimization"]["n_iter"] = 20
    cfg["nn"]["risky_optimization"]["batch_size"] = 32
    cfg["nn"]["risky_optimization"]["log_every"] = 5
    cfg["nn"]["risky_optimization"]["early_stopping"]["enabled"] = False

    # DDP: keep loops short for demo
    cfg["ddp"]["basic_vfi"]["max_iter"] = 120
    cfg["ddp"]["basic_pfi"]["max_iter"] = 20
    cfg["ddp"]["risky_vfi"]["max_iter"] = 5
    cfg["ddp"]["risky_pfi"]["max_iter"] = 5
else:
    cfg["output_root"] = str((DEMO_ROOT / "full").as_posix())

DEMO_CONFIG_PATH = DEMO_ROOT / "configs" / ("part1_baseline_smoke.json" if USE_SMOKE_CONFIG else "part1_baseline_full.json")
DEMO_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
DEMO_CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

print(f"Wrote demo config: {DEMO_CONFIG_PATH}")
print(f"output_root: {cfg['output_root']}")
print(f"training_seed: {TRAINING_SEED}")


## Stage 1: Generate or reuse dataset artifact

This returns a stable `dataset_id`.
If the exact same data config already exists, it reuses the dataset instead of duplicating files.


In [ ]:
raw = run_cli([
    "--config", str(DEMO_CONFIG_PATH),
    "generate-data",
])

data_info = extract_last_json(raw)
dataset_id = data_info["dataset_id"]
print("dataset_id:", dataset_id)


## Stage 2-4: Run DDP + NN + Compare on the same dataset

This demonstrates explicit stage calls (useful for debugging and partial reruns).


In [ ]:
run_id = f"demo-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
print("run_id:", run_id)

raw_ddp = run_cli([
    "--config", str(DEMO_CONFIG_PATH),
    "train-ddp",
    "--run-id", run_id,
    "--dataset-id", dataset_id,
])
ddp_summary = extract_last_json(raw_ddp)

raw_nn = run_cli([
    "--config", str(DEMO_CONFIG_PATH),
    "train-nn",
    "--run-id", run_id,
    "--dataset-id", dataset_id,
])
nn_summary = extract_last_json(raw_nn)

raw_cmp = run_cli([
    "--config", str(DEMO_CONFIG_PATH),
    "compare",
    "--run-id", run_id,
    "--dataset-id", dataset_id,
])
compare_summary = extract_last_json(raw_cmp)

print("Done. Comparison figures:")
print("  basic:", compare_summary["basic"]["figure_path"])
print("  risky:", compare_summary["risky"]["figure_path"])


## Optional shortcut: run everything in one command

`run-all` does all four stages in one shot.


In [ ]:
RUN_ALL_DEMO = False

if RUN_ALL_DEMO:
    raw_all = run_cli([
        "--config", str(DEMO_CONFIG_PATH),
        "run-all",
    ])
    all_summary = extract_last_json(raw_all)
    print(json.dumps(all_summary, indent=2))
else:
    print("Set RUN_ALL_DEMO=True to execute the single-command workflow.")


## Inspect artifacts

Expected structure depends on `run_layout`:

- `run_layout="notebook"` (default):
  - `results/run-.../{data,checkpoints,figures}`
  - `results/latest -> run-...`
- `run_layout="pipeline"`:
  - `results/runs/run-.../{data,checkpoints,figures}`
  - `results/latest -> runs/run-...`

Common:
- `results/datasets/ds_<id>/...` (shared dataset cache)
- `results/latest_dataset -> datasets/ds_<id>`
- `manifest.json` and `run_summary.json` inside each run folder


In [ ]:
output_root = Path(cfg["output_root"])
layout = cfg.get("run_layout", "notebook")
if layout == "notebook":
    run_dir = output_root / run_id
else:
    run_dir = output_root / "runs" / run_id

manifest_path = run_dir / "manifest.json"

print("output_root:", output_root)
print("run_layout:", layout)
print("run_dir:", run_dir)
print("manifest exists:", manifest_path.exists())

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("stages:", list(manifest.get("stages", {}).keys()))


## Safe dataset cleanup (GC)

Dry-run first, then apply only when you are comfortable.


In [ ]:
# Dry-run GC (shows candidates only)
raw_gc = run_cli([
    "--config", str(DEMO_CONFIG_PATH),
    "gc-datasets",
    "--dry-run",
    "--min-age-days", "14",
])
gc_preview = extract_last_json(raw_gc)
print(json.dumps(gc_preview, indent=2))

# Apply deletion (uncomment when ready):
# run_cli([
#     "--config", str(DEMO_CONFIG_PATH),
#     "gc-datasets",
#     "--apply",
#     "--min-age-days", "14",
# ])
